In [0]:
from pyspark.sql import functions as F

df = spark.table("retailnew.bronze.products")

#Cast Types
df = df.withColumn("price", F.col("price").cast("double"))

#Data Quality
df = df.filter(F.col("product_id").isNotNull())
df = df.filter(F.col("price") > 0)

# Standardize category fields
df = (
    df.withColumn("category", F.initcap("category"))
    .withColumn("subcategory", F.initcap("subcategory"))
    .withColumn("brand", F.initcap("brand"))
)

#DeDuplicate
df = df.dropDuplicates(["product_id"])

df = df.select("product_id", "name", "category", "subcategory", "price", "brand", F.current_timestamp().alias("ingest_ts"))

#Write to Silver
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retailnew.silver.products_clean")

)



#Here are the DataFrames in the session:
#- Spark DataFrame 'df', columns
